# 08. P3: Обучение ConvLSTM на ESA CCI как target вместо TTOP

**Цель:** заменить самовычисленный TTOP-таргет на валидированный ESA CCI MAGT (T2m).

**Гипотеза:** ESA CCI ближе к in-situ бурениям, поэтому ESA-trained модель должна показать меньший bias без всякой калибровки.

**Контрольный эксперимент:** меняем ТОЛЬКО target. Архитектура (ConvLSTM v2), 6 фичей climate_core_6 (победитель P4), гиперпараметры — всё то же что в основной модели.

**Что сравниваем:**
- TTOP-trained (P4 winner, climate_core_6 на TTOP target): val RMSE 0.762°C
- ESA-trained (этот ноутбук, climate_core_6 на ESA target): val RMSE = ?
- Обе модели против 33 бурений

In [1]:
# Environment auto-detection
import os
from pathlib import Path

IN_COLAB_VM = (
    'COLAB_RELEASE_TAG' in os.environ or
    'COLAB_GPU' in os.environ
)

if IN_COLAB_VM:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    BASE_DIR = Path('/content/drive/MyDrive/AI4Arctic')
    env_label = 'Colab VM'
else:
    BASE_DIR = Path(os.environ.get(
        'AI4ARCTIC_HOME',
        Path.home() / 'Ai4Arctic'
    ))
    env_label = 'Local runtime'

print(f"Environment: {env_label}")
print(f"BASE_DIR: {BASE_DIR}")
assert BASE_DIR.exists()

import sys
sys.path.insert(0, str(BASE_DIR))

Environment: Local runtime
BASE_DIR: /Users/dariazangirova/Ai4Arctic


In [2]:
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader

if torch.cuda.is_available():
    device = 'cuda'
elif torch.backends.mps.is_available():
    device = 'mps'
else:
    device = 'cpu'
print(f"Device: {device}")

DATA_DIR = BASE_DIR / 'data'
MODELS_DIR = BASE_DIR / 'models'
RESULTS_DIR = BASE_DIR / 'results'
METRICS_DIR = RESULTS_DIR / 'metrics'
FIGURES_DIR = RESULTS_DIR / 'figures'
MAPS_DIR = RESULTS_DIR / 'maps'

for d in [MODELS_DIR, METRICS_DIR, FIGURES_DIR, MAPS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

Device: mps


## 1. Загружаем фичи (X) и сводим к 6 climate_core_6

In [3]:
from src.ablation import FEATURE_NAMES, features_by_names

tensor_npz = np.load(DATA_DIR / 'tensor_01deg_v2.npz')
X_full = tensor_npz['X']                       # (14, 231, 1501, 20)
feature_names = list(tensor_npz['feature_names'])
years_features = list(tensor_npz['years'])     # 2010..2023
print(f"X_full: {X_full.shape}, годы фичей: {years_features}")

# Берём только 6 climate_core_6 (победитель P4)
CORE_6 = ['MAAT', 'LST_winter', 'TDD', 'LST_annual', 'FDD', 'era5_temp']
core_idx = features_by_names(CORE_6)
X_core = X_full[..., core_idx]                  # (14, 231, 1501, 6)
print(f"X_core (только climate_core_6): {X_core.shape}")
print(f"Фичи: {CORE_6}")

X_full: (14, 231, 1501, 20), годы фичей: [np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023)]
X_core (только climate_core_6): (14, 231, 1501, 6)
Фичи: ['MAAT', 'LST_winter', 'TDD', 'LST_annual', 'FDD', 'era5_temp']


## 2. Загружаем уже-готовый ESA CCI target (после reproject)

Файл собран отдельно через block average 0.01° → 0.1°. Содержит T2m на 22 года (2000-2021).

In [4]:
esa_npz = np.load(DATA_DIR / 'esa_cci_target_2000_2021.npz')
y_esa_full = esa_npz['y_esa']                   # (22, 231, 1501)
years_esa = list(esa_npz['years'])              # 2000..2021
print(f"y_esa_full: {y_esa_full.shape}, годы ESA: {years_esa[0]}..{years_esa[-1]}")
print(f"Range: [{np.nanmin(y_esa_full):.2f}, {np.nanmax(y_esa_full):.2f}] °C")
print(f"Медианы по годам (показывает warming trend):")
for i, y in enumerate(years_esa):
    print(f"  {y}: {np.nanmedian(y_esa_full[i]):+.2f}°C")

y_esa_full: (22, 231, 1501), годы ESA: 2000..2021
Range: [-16.56, 6.50] °C
Медианы по годам (показывает warming trend):
  2000: -1.51°C
  2001: -2.27°C
  2002: -1.43°C
  2003: -1.66°C
  2004: -1.48°C
  2005: -1.66°C
  2006: -1.48°C
  2007: -0.97°C
  2008: -0.79°C
  2009: -0.89°C
  2010: -1.36°C
  2011: -1.14°C
  2012: -1.04°C
  2013: -1.36°C
  2014: -0.94°C
  2015: -0.69°C
  2016: -0.70°C
  2017: -0.76°C
  2018: -0.68°C
  2019: -0.69°C
  2020: -0.53°C
  2021: -0.49°C


## 3. Выравниваем годы между фичами и ESA

Фичи: 2010-2023 (14 лет). ESA: 2000-2021 (22 года). Пересечение: 2010-2021 (12 лет).

In [5]:
common_years = sorted(set(years_features) & set(years_esa))
print(f"Общие годы: {common_years} (всего {len(common_years)})")

# Индексы
feat_idx = [years_features.index(y) for y in common_years]
esa_idx = [years_esa.index(y) for y in common_years]

X = X_core[feat_idx]                            # (12, 231, 1501, 6)
y = y_esa_full[esa_idx]                         # (12, 231, 1501)
print(f"После alignment: X={X.shape}, y={y.shape}")
print(f"NaN в y: {np.isnan(y).sum():,} ({100*np.isnan(y).mean():.1f}%)")

Общие годы: [np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021)] (всего 12)
После alignment: X=(12, 231, 1501, 6), y=(12, 231, 1501)
NaN в y: 1,622,484 (39.0%)


## 4. Обучение ConvLSTM на ESA target

Используем готовую `train_one_config` из P4 — она нам подходит, потому что все 6 фичей нужны (нет ablation, только одна конфигурация). 

**Train: годы 2010-2019 (target idx 0..9)**, **Val: 2020-2021 (target idx 10-11)**. Это 2 года валидации, в отличие от P4 (1 год) — потому что у нас больше данных.

In [6]:
from src.ablation import train_one_config
from src.model import ConvLSTMNet

# Индексы целевых лет: train = 4..9 (2014..2019), val = 10..11 (2020..2021)
# Первые 4 года (2010..2013) уйдут в input_window — это нормально
train_targets = list(range(4, 10))    # 6 target years
val_targets = list(range(10, 12))      # 2 target years

print(f"Train targets (year idx): {train_targets} = годы {[common_years[i] for i in train_targets]}")
print(f"Val targets:               {val_targets} = годы {[common_years[i] for i in val_targets]}")

result = train_one_config(
    model_class=ConvLSTMNet,
    X_raw=X,
    y_raw=y,
    feature_indices=list(range(6)),  # все 6 после фильтрации
    train_targets=train_targets,
    val_targets=val_targets,
    epochs=80,
    lr=1e-4,
    batch_size=8,
    device=device,
    early_stop_patience=15,
)

print(f"\n=== Финал ===")
print(f"  val RMSE: {result['val_rmse']:.3f}°C")
print(f"  best epoch: {result['best_epoch']}")
print(f"  time: {result['train_time_sec']/60:.1f} мин")

Train targets (year idx): [4, 5, 6, 7, 8, 9] = годы [np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019)]
Val targets:               [10, 11] = годы [np.int64(2020), np.int64(2021)]
    Train pairs: 1320, Val pairs: 440
    Epoch 0: train=0.4171, val=0.1516, val_rmse=1.701°C
    Epoch 5: train=0.2804, val=0.1234, val_rmse=1.535°C
    Epoch 10: train=0.2538, val=0.1160, val_rmse=1.488°C
    Epoch 15: train=0.2300, val=0.1111, val_rmse=1.456°C
    Epoch 20: train=0.2331, val=0.1281, val_rmse=1.563°C
    Epoch 25: train=0.2109, val=0.1567, val_rmse=1.729°C
    Epoch 30: train=0.1925, val=0.1062, val_rmse=1.424°C
    Epoch 35: train=0.1875, val=0.1132, val_rmse=1.470°C
    Epoch 40: train=0.1753, val=0.1241, val_rmse=1.539°C
    Early stop at epoch 41, best=26, val_rmse=1.598°C

=== Финал ===
  val RMSE: 1.404°C
  best epoch: 26
  time: 4.4 мин


## 5. Сохраняем модель

In [7]:
esa_model_path = MODELS_DIR / 'convlstm_esa_target_climate_core_6.pt'
torch.save({
    'state_dict': result['model_state'],
    'feature_names': CORE_6,
    'val_rmse': result['val_rmse'],
    'y_mean': result['y_mean'],
    'y_std': result['y_std'],
    'best_epoch': result['best_epoch'],
    'description': 'ConvLSTM trained on ESA CCI T2m target, 6 climate features',
    'train_years': [common_years[i] for i in train_targets],
    'val_years': [common_years[i] for i in val_targets],
}, esa_model_path)
print(f"Сохранено: {esa_model_path}")

Сохранено: /Users/dariazangirova/Ai4Arctic/models/convlstm_esa_target_climate_core_6.pt


## 6. Сравнение ESA-trained vs TTOP-trained

Главный вопрос: какая модель ближе к бурениям БЕЗ всякой калибровки?

In [8]:
print('TTOP-trained (P4 winner, climate_core_6 на TTOP target):')
print('  val RMSE на TTOP target: 0.762°C')
print('  но цель TTOP отличается от ESA T2m, прямое сравнение невозможно\n')

print(f'ESA-trained (этот ноутбук, climate_core_6 на ESA target):')
print(f'  val RMSE на ESA T2m target: {result["val_rmse"]:.3f}°C')

print('\nЧтобы корректно сравнить — нужно инференснуть обе модели и оценить против бурений.')
print('Это делается в отдельном сравнительном анализе (см. ноутбук 04 валидация).')

TTOP-trained (P4 winner, climate_core_6 на TTOP target):
  val RMSE на TTOP target: 0.762°C
  но цель TTOP отличается от ESA T2m, прямое сравнение невозможно

ESA-trained (этот ноутбук, climate_core_6 на ESA target):
  val RMSE на ESA T2m target: 1.404°C

Чтобы корректно сравнить — нужно инференснуть обе модели и оценить против бурений.
Это делается в отдельном сравнительном анализе (см. ноутбук 04 валидация).
